# CNN Prediction

Clean 4-class prediction notebook for the exported 224×224 model.

## 1. Imports

In [ ]:
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

## 2. Configuration

In [ ]:
MODEL_PATH = Path("path/to/model")
INPUT_ROOT = Path("path/to/input/images")
SORTED_ROOT = Path("path/to/output/sorted_images")
EXCEL_ROOT = Path("path/to/output")

IMG_HEIGHT = 224
IMG_WIDTH = 224
CLASS_NAMES = [
    "No_Growth",
    "No_PSH",
    "PSH",
    "Contamination",
]

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

## 3. Load model

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)

if model.output_shape[-1] != len(CLASS_NAMES):
    raise ValueError(
        f"Model has {model.output_shape[-1]} output classes, "
        f"but CLASS_NAMES has {len(CLASS_NAMES)}."
    )

print(f"Loaded model: {MODEL_PATH}")

## 4. Find images

In [ ]:
def natural_sort_key(path):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", str(path))
    ]


def find_images(folder):
    images = [
        path
        for path in Path(folder).rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]
    return sorted(images, key=natural_sort_key)

## 5. Preprocess image

In [ ]:
def preprocess_image(image_path):
    image = tf.keras.utils.load_img(
        image_path,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
    )
    image = tf.keras.utils.img_to_array(image)
    image = tf.image.adjust_contrast(image, 2.0)
    image = image / 255.0
    return image

## 6. Predict images

In [ ]:
def predict_images(image_paths):
    rows = []

    for image_path in image_paths:
        image = preprocess_image(image_path)
        probabilities = model.predict(
            np.expand_dims(image, axis=0),
            verbose=0,
        )[0]

        class_index = int(np.argmax(probabilities))
        rows.append({
            "File Name": image_path.name,
            "Predicted Class": CLASS_NAMES[class_index],
            "Confidence": float(probabilities[class_index]),
            "Source Path": str(image_path),
        })

    return pd.DataFrame(rows)

## 7. Save predictions

In [ ]:
def save_predictions(predictions, source_folder, batch_name):
    source_folder = Path(source_folder)
    batch_output = SORTED_ROOT / batch_name

    for row in predictions.itertuples(index=False):
        source_path = Path(row[3])
        predicted_class = row[1]

        destination = batch_output / predicted_class / source_path.relative_to(source_folder)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, destination)

    EXCEL_ROOT.mkdir(parents=True, exist_ok=True)
    excel_path = EXCEL_ROOT / f"{batch_name}_predictions.xlsx"
    predictions.drop(columns="Source Path").to_excel(excel_path, index=False)
    print(f"Saved: {excel_path}")

## 8. Predict each subfolder

In [ ]:
SORTED_ROOT.mkdir(parents=True, exist_ok=True)

subfolders = sorted(
    [path for path in INPUT_ROOT.iterdir() if path.is_dir()],
    key=natural_sort_key,
)

for subfolder in subfolders:
    image_paths = find_images(subfolder)
    print(f"{subfolder.name}: {len(image_paths)} images")

    predictions = predict_images(image_paths)
    save_predictions(predictions, subfolder, subfolder.name)